# Rayleigh–Bénard Convection

This notebook compares MI-DL, SI-DL, and the known Rayleigh-number coordinate for turbulent Rayleigh–Bénard convection.

The dimensional inputs are height, temperature difference, thermal conductivity, gravity, thermal expansion, kinematic viscosity, and thermal diffusivity. The response is the Nusselt number. The search identifies a dimensionless coordinate and evaluates it with mutual information, $S_{cov}$, and held-out GPR error.


## 1. Imports and experiment settings

Load the local MI-DL and SI-DL implementations, define the dimensional matrix, and configure the repeated differential-evolution searches. The source dataset is stored locally in `data/benard.csv`.


In [ ]:
from __future__ import annotations

import os

import sys

import warnings

from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/matplotlib-codex-cache")

os.environ.setdefault("MPLBACKEND", "Agg")

warnings.filterwarnings("ignore", category=RuntimeWarning)

import matplotlib.pyplot as plt

import numpy as np

import pandas as pd

from matplotlib.ticker import ScalarFormatter

from scipy.optimize import differential_evolution

from sklearn.gaussian_process import GaussianProcessRegressor

from sklearn.gaussian_process.kernels import ConstantKernel, RBF, WhiteKernel

from sklearn.metrics import mean_squared_error

from sklearn.model_selection import train_test_split

from sklearn.pipeline import make_pipeline

from sklearn.preprocessing import StandardScaler

OUTPUT_DIR = Path.cwd()

ROOT = OUTPUT_DIR.parent

for module_dir in [ROOT / "Compare" / "MI-DL", ROOT / "SI-DL-main"]:
    if str(module_dir) not in sys.path:
        sys.path.insert(0, str(module_dir))

import midl

import SI_DL

RANDOM_STATE = 42

FIG_DIR = OUTPUT_DIR / "figures"

CSV_DIR = OUTPUT_DIR / "csv"

DATA_PATH = OUTPUT_DIR / "data" / "benard.csv"

INPUT_COLUMNS = ["h", "delta_T", "lambda", "g", "alpha", "v", "k"]

VARIABLE_LABELS = ["h", "delta_T", "lambda", "g", "alpha", "nu", "kappa"]

MI_K_NEIGHBORS = 6

MI_DE_MAXITER = 180

MI_RESTART_SEEDS = [
    0,
    1,
    2,
    3,
    4,
    5,
    7,
    11,
    13,
    17,
    19,
    23,
    31,
    37,
    42,
    57,
    73,
    89,
    101,
    137,
    173,
    202,
    251,
    307,
]

SI_BANDWIDTH = 0.097

SI_MAXITER = 1000

SI_POPSIZE = 16

SI_RESTART_SEEDS = [0, 1, 2, 3, 4, 5, 7, 11, 17, 23, 42, 101, 202]

TEST_SIZE = 0.25

D_IN = np.array(
    [
        [1, 0, 1, 1, 0, 2, 2],
        [0, 0, -3, -2, 0, -1, -1],
        [0, 0, 1, 0, 0, 0, 0],
        [0, 1, -1, 0, -1, 0, 0],
    ],
    dtype=float,
)

PHYSICAL_BASIS = np.array(
    [
        [0, 0, 0, 0, 0, 1, -1],
        [0, 1, 0, 0, 1, 0, 0],
        [3, 0, 0, 1, 0, -2, 0],
    ],
    dtype=float,
)

KNOWN_RA_EXPONENTS = np.array(
    [1.0, 1.0 / 3.0, 0.0, 1.0 / 3.0, 1.0 / 3.0, -1.0 / 3.0, -1.0 / 3.0],
    dtype=float,
)


## 2. Data and dimensionless-coordinate utilities

Load the positive physical measurements, construct the Nusselt response, normalize exponent vectors, and evaluate information-based metrics.


In [ ]:
def load_benard_data() -> tuple[pd.DataFrame, np.ndarray, np.ndarray]:
    df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")
    q = df["q"].to_numpy(float)
    h = df["h"].to_numpy(float)
    delta_t = df["delta_T"].to_numpy(float)
    lambda_ = df["lambda"].to_numpy(float)
    X = df[INPUT_COLUMNS].to_numpy(float)
    Y = (q * h) / (lambda_ * delta_t)
    if np.any(X <= 0.0) or np.any(Y <= 0.0):
        raise ValueError("Benard inputs and Nu must be positive.")
    return df, X, Y

def normalize_exponents(exponents: np.ndarray) -> np.ndarray:
    row = np.asarray(exponents, dtype=float).reshape(-1)
    scale = float(np.max(np.abs(row)))
    if scale <= 1e-12:
        return row
    out = row / scale
    first = np.flatnonzero(np.abs(out) > 1e-10)
    if first.size and out[first[0]] < 0.0:
        out = -out
    return out

def make_pi_from_exponents(X: np.ndarray, exponents: np.ndarray) -> np.ndarray:
    return np.exp(np.log(X) @ np.asarray(exponents, dtype=float).reshape(-1))

def formula_from_exponents(exponents: np.ndarray, decimals: int = 4) -> str:
    terms = []
    for label, value in zip(VARIABLE_LABELS, np.asarray(exponents).reshape(-1)):
        value = float(value)
        if abs(value) < 5e-5:
            continue
        terms.append(f"{label}^{value:.{decimals}f}")
    return " * ".join(terms)

def information_metrics(feature: np.ndarray, Y: np.ndarray) -> dict:
    info = midl.information_lower_bound(
        np.asarray(feature, dtype=float).reshape(-1, 1),
        Y,
        k_neighbors=MI_K_NEIGHBORS,
        random_state=RANDOM_STATE,
    )
    return {
        "mutual_information": float(info["mutual_information"]),
        "epsilon_lb_normalized": float(info["epsilon_lb"] / np.var(Y, ddof=0)),
    }


## 3. MI-DL search

Search the null space of the dimensional matrix with multiple differential-evolution restarts. The best restart maximizes mutual information between the learned coordinate and the Nusselt response.


In [ ]:
def run_midl_raw_search(X: np.ndarray, Y: np.ndarray) -> dict:
    basis, _ = midl.calc_basis(D_IN)
    rows = []
    best = None

    def objective(params: np.ndarray) -> float:
        raw_exponents = np.asarray(basis @ np.asarray(params, dtype=float).reshape(-1)).reshape(-1)
        exponents = normalize_exponents(raw_exponents)
        if np.max(np.abs(exponents)) <= 1e-12:
            return 1e6
        try:
            pi = make_pi_from_exponents(X, exponents)
            if not np.all(np.isfinite(pi)) or np.any(pi <= 0.0):
                return 1e6
            mi = midl._knn_mutual_information(
                pi.reshape(-1, 1),
                Y,
                n_neighbors=MI_K_NEIGHBORS,
                random_state=RANDOM_STATE,
            )
        except Exception:
            return 1e6
        if not np.isfinite(mi):
            return 1e6
        return -float(mi)

    for seed in MI_RESTART_SEEDS:
        result = differential_evolution(
            objective,
            bounds=[(-1.0, 1.0)] * basis.shape[1],
            strategy="best1bin",
            maxiter=MI_DE_MAXITER,
            popsize=10,
            seed=seed,
            polish=False,
            updating="deferred",
            workers=1,
        )
        raw_exponents = np.asarray(basis @ result.x).reshape(-1)
        exponents = normalize_exponents(raw_exponents)
        pi = make_pi_from_exponents(X, exponents)
        mi_raw = information_metrics(pi, Y)
        row = {
            "seed": seed,
            "mi_raw_pi": mi_raw["mutual_information"],
            "epsilon_lb_raw_pi_normalized": mi_raw["epsilon_lb_normalized"],
            "formula": formula_from_exponents(exponents),
            "exponents": exponents,
            "pi": pi,
            "optimizer_result": result,
        }
        rows.append(row)
        if best is None or row["mi_raw_pi"] > best["mi_raw_pi"]:
            best = row
    audit = pd.DataFrame([{k: v for k, v in row.items() if k not in {"exponents", "pi", "optimizer_result"}} for row in rows])
    audit = audit.sort_values("mi_raw_pi", ascending=False)
    return {
        "exponents": best["exponents"],
        "pi": best["pi"],
        "best_seed": int(best["seed"]),
        "search_score": float(best["mi_raw_pi"]),
        "optimizer_result": best["optimizer_result"],
        "restart_audit": audit,
    }


## 4. SI-DL search

Optimize the covariance-based SI-DL objective using the physical dimensionless basis and a fixed kernel bandwidth.


In [ ]:
def params_to_pi(params: np.ndarray, X: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    exponents = normalize_exponents(np.asarray(params, dtype=float).reshape(-1) @ PHYSICAL_BASIS)
    return exponents, make_pi_from_exponents(X, exponents)

def run_sidl_search(X: np.ndarray, Y: np.ndarray) -> dict:
    bounds = [(-2.0, 2.0)] * PHYSICAL_BASIS.shape[0]

    def objective(params: np.ndarray) -> float:
        try:
            exponents, pi = params_to_pi(params, X)
            if not np.all(np.isfinite(pi)) or np.any(pi <= 0.0):
                return 1e6
            score = SI_DL.explained_variance_score(pi, Y, bandwidth=SI_BANDWIDTH)["S_cov"]
        except Exception:
            return 1e6
        if not np.isfinite(score):
            return 1e6
        return -float(score)

    rows = []
    best = None
    for seed in SI_RESTART_SEEDS:
        result = differential_evolution(
            objective,
            bounds=bounds,
            maxiter=SI_MAXITER,
            popsize=SI_POPSIZE,
            seed=seed,
            polish=True,
            updating="immediate",
            workers=1,
        )
        exponents, pi = params_to_pi(result.x, X)
        score = SI_DL.explained_variance_score(pi, Y, bandwidth=SI_BANDWIDTH)
        row = {
            "seed": seed,
            "S_cov": float(score["S_cov"]),
            "sidl_error": float(1.0 - score["S_cov"]),
            "formula": formula_from_exponents(exponents),
            "exponents": exponents,
            "pi": pi,
            "optimizer_result": result,
            "score": score,
        }
        rows.append(row)
        if best is None or row["S_cov"] > best["S_cov"]:
            best = row

    audit = pd.DataFrame([{k: v for k, v in row.items() if k not in {"exponents", "pi", "optimizer_result", "score"}} for row in rows])
    audit = audit.sort_values("S_cov", ascending=False)
    return {
        "optimizer_result": best["optimizer_result"],
        "exponents": best["exponents"],
        "pi": best["pi"],
        "best_seed": int(best["seed"]),
        "search_score": float(best["S_cov"]),
        "score": best["score"],
        "restart_audit": audit,
    }


## 5. GPR evaluation and common scores

Fit the same one-dimensional GPR model to each candidate coordinate and compute held-out prediction error together with information and covariance scores.


In [ ]:
def gpr_fit(feature: np.ndarray, Y: np.ndarray, train_idx: np.ndarray, test_idx: np.ndarray) -> dict:
    feature = np.asarray(feature, dtype=float).reshape(-1, 1)
    Y = np.asarray(Y, dtype=float).reshape(-1)
    kernel = ConstantKernel(1.0, (1e-3, 1e3)) * RBF(1.0, (1e-3, 1e3))
    kernel += WhiteKernel(1.0, (1e-8, 1e3))
    model = make_pipeline(
        StandardScaler(),
        GaussianProcessRegressor(
            kernel=kernel,
            alpha=1e-8,
            normalize_y=True,
            n_restarts_optimizer=8,
            random_state=RANDOM_STATE,
        ),
    )
    model.fit(feature[train_idx], Y[train_idx])
    y_pred = model.predict(feature[test_idx])
    mse_raw = float(mean_squared_error(Y[test_idx], y_pred))
    scale = float(np.var(Y[train_idx], ddof=0))
    return {
        "model": model,
        "y_pred_test": y_pred,
        "mse_raw": mse_raw,
        "mse_normalized": mse_raw / scale,
        "rmse_normalized": float(np.sqrt(mse_raw / scale)),
        "error_scale_var_y_train": scale,
    }

def score_metrics(feature: np.ndarray, Y: np.ndarray) -> dict:
    sidl_score = SI_DL.explained_variance_score(feature, Y, bandwidth=SI_BANDWIDTH)
    mi_raw = information_metrics(feature, Y)
    return {
        **mi_raw,
        "S_cov": float(sidl_score["S_cov"]),
        "sidl_error": float(1.0 - sidl_score["S_cov"]),
        "sidl_bandwidth": float(sidl_score["bandwidth"]),
    }


## 6. Figures

Create the GPR-fit comparison and a compact summary table. Both images are written to `figures/`.


In [ ]:
def use_thousands_axis(ax) -> None:
    """Show large Benard axes with mathtext scientific notation."""
    for axis in (ax.xaxis, ax.yaxis):
        formatter = ScalarFormatter(useMathText=True)
        formatter.set_powerlimits((3, 3))
        axis.set_major_formatter(formatter)

def plot_gpr_fit(coordinates: dict, Y: np.ndarray, train_idx: np.ndarray, test_idx: np.ndarray, fits: dict) -> None:
    methods = [
        ("MI-DL", r"MI-DL $\mathit{\Pi}_{1}$"),
        ("SI-DL", r"SI-DL $\mathit{\Pi}_{1}$"),
        ("Known Ra", r"Known Ra $\mathit{\Pi}_{1}$"),
    ]
    plt.rcParams.update(
        {
            "font.family": "STIXGeneral",
            "mathtext.fontset": "stix",
            "axes.titlesize": 22,
            "axes.labelsize": 21,
            "xtick.labelsize": 17,
            "ytick.labelsize": 17,
            "legend.fontsize": 15,
        }
    )
    fig, axes = plt.subplots(1, 3, figsize=(17.2, 5.7), sharey=True, dpi=260)
    for ax, method in zip(axes, methods):
        method_name, xlabel = method
        feature = coordinates[method_name]["pi"]
        fit = fits[method_name]
        line_x = np.linspace(feature.min(), feature.max(), 320).reshape(-1, 1)
        y_mean = fit["model"].predict(line_x)
        ax.scatter(
            feature,
            Y,
            s=42,
            alpha=0.58,
            color="#1f77b4",
            edgecolors="white",
            linewidths=0.45,
            label="data",
            zorder=2,
        )
        ax.plot(line_x[:, 0], y_mean, color="#d62728", linewidth=2.9, label="GPR mean", zorder=3)
        ax.text(
            0.97,
            0.06,
            f"GPR norm. MSE = {fit['mse_normalized']:.4f}",
            transform=ax.transAxes,
            ha="right",
            va="bottom",
            fontsize=15,
            bbox={
                "boxstyle": "round,pad=0.24",
                "facecolor": "white",
                "edgecolor": "#d1d5db",
                "alpha": 0.92,
            },
        )
        ax.set_xlabel(xlabel)
        ax.set_title(method_name)
        ax.grid(True, alpha=0.22, linewidth=0.9)
        use_thousands_axis(ax)
        ax.legend(
            frameon=True,
            facecolor="white",
            edgecolor="#d1d5db",
            framealpha=0.94,
            loc="upper left",
            handlelength=2.2,
            borderpad=0.65,
        )
    axes[0].set_ylabel("Nu")
    fig.suptitle("Benard", fontsize=25, y=1.02)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "gpr_fit.png", bbox_inches="tight", facecolor="white")
    plt.close(fig)

def plot_summary_table(summary: pd.DataFrame) -> None:
    fig, ax = plt.subplots(figsize=(17.0, 4.6), dpi=220)
    ax.axis("off")
    fig.patch.set_facecolor("white")
    ax.text(0.5, 0.96, "Benard: MI-DL vs SI-DL vs Known Ra", ha="center", va="top", fontsize=18, weight="bold")
    rows = []
    for _, row in summary.iterrows():
        rows.append(
            [
                row["method"],
                row["formula"],
                f"{row['mutual_information']:.4f}",
                f"{row['epsilon_lb_normalized']:.5f}",
                f"{row['S_cov']:.4f}",
                f"{row['gpr_mse_normalized']:.5f}",
            ]
        )
    table = ax.table(
        cellText=rows,
        colLabels=["Method", "Formula", "MI k=6", "epsilon_LB/Var", "S_cov", "GPR norm. MSE"],
        cellLoc="center",
        colLoc="center",
        colWidths=[0.11, 0.51, 0.09, 0.10, 0.09, 0.10],
        bbox=[0.02, 0.08, 0.96, 0.78],
    )
    table.auto_set_font_size(False)
    for (r, c), cell in table.get_celld().items():
        cell.set_edgecolor("#3f3f46")
        cell.set_linewidth(0.85)
        cell.PAD = 0.03
        text = cell.get_text()
        if r == 0:
            cell.set_facecolor("#1f2937")
            text.set_color("white")
            text.set_weight("bold")
            text.set_fontsize(10.5)
        else:
            cell.set_facecolor("#f8fafc" if r % 2 == 1 else "#ffffff")
            text.set_fontsize(7.2 if c == 1 else 10.0)
            if c == 1:
                text.set_ha("left")
            if c == 0:
                text.set_weight("bold")
    fig.savefig(FIG_DIR / "summary.png", bbox_inches="tight", facecolor="white")
    plt.close(fig)


## 7. Run the comparison

Execute MI-DL and SI-DL, compare them with the known Rayleigh coordinate, save five result tables to `csv/`, and create the two figures.


In [ ]:
def run_comparison() -> dict:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    CSV_DIR.mkdir(parents=True, exist_ok=True)
    df, X, Y = load_benard_data()

    midl_result = run_midl_raw_search(X, Y)
    sidl_result = run_sidl_search(X, Y)
    coordinates = {
        "MI-DL": {"pi": midl_result["pi"], "exponents": midl_result["exponents"]},
        "SI-DL": {"pi": sidl_result["pi"], "exponents": sidl_result["exponents"]},
        "Known Ra": {
            "pi": make_pi_from_exponents(X, KNOWN_RA_EXPONENTS),
            "exponents": KNOWN_RA_EXPONENTS,
        },
    }

    train_idx, test_idx = train_test_split(np.arange(X.shape[0]), test_size=TEST_SIZE, random_state=RANDOM_STATE)
    fits = {}
    summary_rows = []
    for method, values in coordinates.items():
        fit = gpr_fit(values["pi"], Y, train_idx, test_idx)
        fits[method] = fit
        common = score_metrics(values["pi"], Y)
        summary_rows.append(
            {
                "method": method,
                "formula": formula_from_exponents(values["exponents"]),
                "exponents": values["exponents"],
                **common,
                "gpr_mse_raw": fit["mse_raw"],
                "gpr_mse_normalized": fit["mse_normalized"],
                "gpr_rmse_normalized": fit["rmse_normalized"],
                "best_seed": (
                    midl_result["best_seed"]
                    if method == "MI-DL"
                    else sidl_result["best_seed"]
                    if method == "SI-DL"
                    else np.nan
                ),
            }
        )
    summary = pd.DataFrame(summary_rows)
    summary["rank_by_MI"] = summary["mutual_information"].rank(ascending=False, method="min").astype(int)
    summary["rank_by_gpr"] = summary["gpr_mse_normalized"].rank(method="min").astype(int)
    summary["rank_by_S_cov"] = summary["S_cov"].rank(ascending=False, method="min").astype(int)

    summary_csv = summary.copy()
    for j, label in enumerate(VARIABLE_LABELS):
        summary_csv[f"pi1_exp_{label}"] = summary_csv["exponents"].map(lambda arr, jj=j: arr[jj])
    summary_csv.drop(columns=["exponents"]).to_csv(CSV_DIR / "summary.csv", index=False)
    midl_result["restart_audit"].to_csv(CSV_DIR / "midl_restarts.csv", index=False)
    sidl_result["restart_audit"].to_csv(CSV_DIR / "sidl_restarts.csv", index=False)
    pd.DataFrame(
        {
            "Nu": Y,
            "MI-DL_raw_pi1": coordinates["MI-DL"]["pi"],
            "SI-DL_raw_pi1": coordinates["SI-DL"]["pi"],
            "Known_Ra_raw_pi1": coordinates["Known Ra"]["pi"],
            "source": df["source"].to_numpy(),
        }
    ).to_csv(CSV_DIR / "coordinates.csv", index=False)

    exponent_rows = []
    for method, values in coordinates.items():
        for label, exponent in zip(VARIABLE_LABELS, values["exponents"]):
            exponent_rows.append(
                {
                    "method": method,
                    "variable": label,
                    "normalized_exponent": float(exponent),
                }
            )
    pd.DataFrame(exponent_rows).to_csv(CSV_DIR / "exponents.csv", index=False)

    plot_gpr_fit(coordinates, Y, train_idx, test_idx, fits)
    plot_summary_table(summary)
    return {"summary": summary, "fits": fits}


In [ ]:
outputs = run_comparison()
outputs["summary"][[
    "method",
    "formula",
    "mutual_information",
    "epsilon_lb_normalized",
    "S_cov",
    "gpr_mse_normalized",
    "rank_by_MI",
    "rank_by_gpr",
]]
